# Exploración Climática de Ciudades del Perú

Datos diarios de temperatura y precipitación (2023) via Open-Meteo API

In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

PALETA = ['#264653', '#2A9D8F', '#E9C46A', '#F4A261', '#E76F51', '#606C38']
sns.set_palette(PALETA)

## Extracción de datos desde Open-Meteo

In [ ]:
ciudades = {
    "Lima": (-12.05, -77.04),
    "Arequipa": (-16.40, -71.54),
    "Cusco": (-13.52, -71.97),
    "Iquitos": (-3.75, -73.25),
    "Puno": (-15.84, -70.02),
    "Piura": (-5.19, -80.63),
}

frames = []
for ciudad, (lat, lon) in ciudades.items():
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": "2023-01-01", "end_date": "2023-12-31",
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "timezone": "America/Lima"
    }
    resp = requests.get(url, params=params, timeout=30)
    data = resp.json()["daily"]
    df = pd.DataFrame(data)
    df["ciudad"] = ciudad
    frames.append(df)

clima = pd.concat(frames, ignore_index=True)
clima["time"] = pd.to_datetime(clima["time"])
clima["amplitud_termica"] = clima["temperature_2m_max"] - clima["temperature_2m_min"]
clima["mes"] = clima["time"].dt.month
clima.shape

In [ ]:
clima.head(10)

In [ ]:
clima.groupby('ciudad')[['temperature_2m_max', 'temperature_2m_min', 'precipitation_sum']].describe().round(1)

## Temperatura máxima promedio mensual por ciudad

In [ ]:
mensual_temp = clima.groupby(['ciudad', 'mes'])['temperature_2m_max'].mean().reset_index()
meses_labels = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

fig, ax = plt.subplots(figsize=(14, 6))
for i, ciudad in enumerate(ciudades.keys()):
    datos = mensual_temp[mensual_temp['ciudad'] == ciudad]
    ax.plot(datos['mes'], datos['temperature_2m_max'], marker='o', markersize=5,
            label=ciudad, color=PALETA[i], linewidth=2)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(meses_labels)
ax.set_title('Temperatura Máxima Promedio Mensual por Ciudad (2023)', fontsize=14, fontweight='bold')
ax.set_xlabel('Mes')
ax.set_ylabel('Temperatura (°C)')
ax.legend(loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.show()

## Distribución de temperatura máxima por ciudad

In [ ]:
orden_ciudades = clima.groupby('ciudad')['temperature_2m_max'].median().sort_values().index.tolist()

fig, ax = plt.subplots(figsize=(14, 6))
sns.violinplot(data=clima, x='ciudad', y='temperature_2m_max', order=orden_ciudades,
               palette=PALETA, inner='quartile', ax=ax, linewidth=1.2)
ax.set_title('Distribución de Temperatura Máxima Diaria por Ciudad (2023)', fontsize=14, fontweight='bold')
ax.set_xlabel('Ciudad')
ax.set_ylabel('Temperatura Máxima (°C)')
plt.tight_layout()
plt.show()

## Precipitación mensual: Lima vs Iquitos vs Cusco

In [ ]:
ciudades_precip = ['Lima', 'Iquitos', 'Cusco']
precip_mensual = (clima[clima['ciudad'].isin(ciudades_precip)]
                  .groupby(['ciudad', 'mes'])['precipitation_sum']
                  .sum().reset_index())

colores_precip = {'Lima': '#264653', 'Iquitos': '#2A9D8F', 'Cusco': '#E76F51'}
x = np.arange(12)
ancho = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
for i, ciudad in enumerate(ciudades_precip):
    datos = precip_mensual[precip_mensual['ciudad'] == ciudad].sort_values('mes')
    ax.bar(x + i * ancho, datos['precipitation_sum'].values, ancho,
           label=ciudad, color=colores_precip[ciudad], edgecolor='white', linewidth=0.5)

ax.set_xticks(x + ancho)
ax.set_xticklabels(meses_labels)
ax.set_title('Precipitación Mensual Acumulada: Lima vs Iquitos vs Cusco (2023)', fontsize=14, fontweight='bold')
ax.set_xlabel('Mes')
ax.set_ylabel('Precipitación (mm)')
ax.legend()
plt.tight_layout()
plt.show()

## Temperatura diaria por ciudad (interactivo)

In [ ]:
fig = px.line(clima, x='time', y='temperature_2m_max', color='ciudad',
              title='Temperatura Máxima Diaria por Ciudad (2023)',
              labels={'temperature_2m_max': 'Temp. Máxima (°C)', 'time': 'Fecha', 'ciudad': 'Ciudad'},
              color_discrete_sequence=PALETA,
              template='plotly_white')
fig.update_layout(
    hovermode='x unified',
    legend=dict(orientation='h', y=-0.15),
    xaxis=dict(rangeslider=dict(visible=True), type='date')
)
fig.update_traces(line=dict(width=1.2))
fig.show()

## Temperatura promedio mensual por ciudad (heatmap interactivo)

In [ ]:
pivot_temp = clima.groupby(['ciudad', 'mes'])['temperature_2m_max'].mean().reset_index()
pivot_wide = pivot_temp.pivot(index='ciudad', columns='mes', values='temperature_2m_max')
pivot_wide.columns = meses_labels

fig = px.imshow(pivot_wide.round(1),
                title='Temperatura Máxima Promedio Mensual por Ciudad',
                labels=dict(x='Mes', y='Ciudad', color='Temp. (°C)'),
                color_continuous_scale='YlOrRd',
                text_auto='.1f',
                template='plotly_white',
                aspect='auto')
fig.update_layout(height=400)
fig.show()

## Resumen por ciudad

In [ ]:
resumen = clima.groupby('ciudad').agg(
    temp_max_media=('temperature_2m_max', 'mean'),
    temp_min_media=('temperature_2m_min', 'mean'),
    amplitud_media=('amplitud_termica', 'mean'),
    precip_total=('precipitation_sum', 'sum'),
    dias_con_lluvia=('precipitation_sum', lambda x: (x > 0.1).sum()),
    registros=('time', 'count')
).round(1)
resumen.sort_values('temp_max_media', ascending=False)